# Azure AI Search - Query Script

This notebook demonstrates different types of searches on indexed documents using Azure AI Search.

## Search Types Covered

| Search Type | Description | Best For |
|-------------|-------------|----------|
| **Vector Similarity Search** | Pure vector-based semantic search using embeddings | Semantic meaning, concepts |
| **Hybrid Search** | Combines vector and keyword (BM25) search | Balance of keywords + meaning |
| **Hybrid + Semantic Reranking** | Hybrid search with AI-powered reranking | Highest quality results |

## Prerequisites

Before running this notebook, ensure you have:
1. Completed the `setup_search_index.ipynb` notebook
2. Verified the indexer has finished processing documents
3. A `.env` file with the required environment variables

Based on: [Azure Search Vector Samples](https://github.com/Azure/azure-search-vector-samples)

## Step 1: Import Required Libraries

We import:
- **SearchClient**: For executing search queries against the index
- **VectorizedQuery**: For creating vector search queries
- **QueryType, QueryCaptionType, QueryAnswerType**: For semantic search configuration
- **AzureOpenAI**: For generating query embeddings

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import (
    VectorizedQuery,
    QueryType,
    QueryCaptionType,
    QueryAnswerType
)
from openai import AzureOpenAI

# Load environment variables
load_dotenv(dotenv_path="../.env")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Configuration and Environment Variables

Load the configuration for connecting to Azure AI Search and Azure OpenAI services.

In [2]:
# Configuration
INDEX_NAME = "student-loan-guide-index"

# Get environment variables
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_AI_SERVICES_ENDPOINT")
AZURE_SEARCH_KEY = os.getenv("AZURE_AI_SERVICES_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_ADA002_EMBEDDING_DEPLOYMENT")

# Validate environment variables
required_vars = {
    "AZURE_AI_SERVICES_ENDPOINT": AZURE_SEARCH_ENDPOINT,
    "AZURE_AI_SERVICES_KEY": AZURE_SEARCH_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_KEY": AZURE_OPENAI_KEY,
    "AZURE_OPENAI_ADA002_EMBEDDING_DEPLOYMENT": AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
}

missing_vars = [var for var, value in required_vars.items() if not value]

if missing_vars:
    print("❌ Missing required environment variables:")
    for var in missing_vars:
        print(f"   - {var}")
else:
    print("✅ All environment variables loaded!")
    print(f"\n📋 Configuration:")
    print(f"   - Index Name: {INDEX_NAME}")
    print(f"   - Search Endpoint: {AZURE_SEARCH_ENDPOINT}")
    print(f"   - OpenAI Endpoint: {AZURE_OPENAI_ENDPOINT}")

✅ All environment variables loaded!

📋 Configuration:
   - Index Name: student-loan-guide-index
   - Search Endpoint: https://ai102srch193837986.search.windows.net
   - OpenAI Endpoint: https://general-ai-projects-resource.openai.azure.com/


## Step 3: Initialize Clients

Create the search client for querying the index and the OpenAI client for generating embeddings.

In [3]:
# Initialize Search Client
credential = AzureKeyCredential(AZURE_SEARCH_KEY)
search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

# Initialize Azure OpenAI Client
openai_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version="2024-02-01"
)

print("✅ Clients initialized!")

✅ Clients initialized!


## Step 4: Define Helper Functions

### Embedding Function
Generates vector embeddings for search queries using Azure OpenAI's text-embedding-ada-002 model.

In [4]:
def get_embedding(text: str) -> list:
    """
    Generate embeddings for a text query using Azure OpenAI
    
    Args:
        text (str): The text to generate embeddings for
    
    Returns:
        list: The embedding vector (1536 dimensions)
    """
    response = openai_client.embeddings.create(
        input=text,
        model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    )
    
    return response.data[0].embedding

# Test the embedding function
test_embedding = get_embedding("test query")
print(f"✅ Embedding function working! Vector dimension: {len(test_embedding)}")

✅ Embedding function working! Vector dimension: 1536


### Results Printing Function
Formats and displays search results in a readable format.

In [5]:
def print_search_results(results, search_type: str):
    """
    Print search results in a formatted way
    
    Args:
        results: Search results from Azure AI Search
        search_type (str): Type of search performed
    """
    print("\n" + "=" * 100)
    print(f"📊 {search_type} Results")
    print("=" * 100)
    
    result_count = 0
    for result in results:
        result_count += 1
        score = result['@search.score']
        chunk = result.get('chunk', 'N/A')
        title = result.get('title', 'N/A')
        
        print(f"\n🔍 Result #{result_count} (Score: {score:.4f})")
        print(f"   📄 Title: {title}")
        print(f"   📝 Content: {chunk[:300]}..." if len(chunk) > 300 else f"   📝 Content: {chunk}")
        
        # Print captions if available (for semantic search)
        if '@search.captions' in result:
            captions = result['@search.captions']
            if captions:
                print(f"   💡 Caption: {captions[0].text}")
        
        # Print reranker score if available
        if '@search.reranker_score' in result:
            reranker_score = result['@search.reranker_score']
            if reranker_score is not None:
                print(f"   🎯 Reranker Score: {reranker_score:.4f}")
        
        print("-" * 100)
    
    if result_count == 0:
        print("\n❌ No results found")
    else:
        print(f"\n✅ Total results: {result_count}")

print("✅ Print function defined!")

✅ Print function defined!


## Step 5: Vector Similarity Search

**Pure semantic vector search** using embeddings.

### How it Works
1. Convert the query text into a vector embedding
2. Find the K nearest neighbors in the vector space
3. Return documents with the most similar vectors

### Best For
- Understanding semantic meaning
- Finding conceptually related content
- Queries where exact keywords may not appear in documents

In [6]:
def vector_similarity_search(query: str, top_k: int = 5):
    """
    Perform a pure vector similarity search
    
    Args:
        query (str): The search query
        top_k (int): Number of results to return
    """
    print(f"\n🔍 Performing Vector Similarity Search for: '{query}'")
    print("   Method: Pure semantic vector search using embeddings")
    
    # Generate query embedding
    query_vector = get_embedding(query)
    
    # Create vector query
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=top_k,
        fields="vector"
    )
    
    # Perform search
    results = search_client.search(
        search_text=None,  # No text search, pure vector
        vector_queries=[vector_query],
        select=["chunk_id", "chunk", "title", "parent_id"],
        top=top_k
    )
    
    print_search_results(results, "Vector Similarity Search")

print("✅ Vector similarity search function defined!")

✅ Vector similarity search function defined!


### Try Vector Similarity Search

Run a vector search with a sample query:

In [7]:
# Example vector similarity search
vector_similarity_search("What are the eligibility requirements for student loans?", top_k=3)


🔍 Performing Vector Similarity Search for: 'What are the eligibility requirements for student loans?'
   Method: Pure semantic vector search using embeddings

📊 Vector Similarity Search Results

🔍 Result #1 (Score: 0.8844)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: How to qualify for private student loans 

Private student loans are primarily offered by banks, credit unions, and online lenders. They 
usually require a credit check to evaluate your creditworthiness. Most private lenders will 
let you apply with a cosigner if you don't meet their specific criter...
----------------------------------------------------------------------------------------------------

🔍 Result #2 (Score: 0.8794)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: loans are another option. Both have similar 
eligibility requirements, but they also have plenty of important differences you should 
know. 

These are the eligibility requirement for federal and private student loans: 

 Federal student loans Priv

## Step 6: Hybrid Search

**Combines vector similarity and keyword (BM25) search** for better results.

### How it Works
1. Performs both vector similarity search AND traditional keyword search
2. Uses Reciprocal Rank Fusion (RRF) to combine results
3. Returns documents that score well on both approaches

### Best For
- General-purpose search
- Queries with both semantic meaning AND specific keywords
- Balancing exact matches with conceptual relevance

In [8]:
def hybrid_search(query: str, top_k: int = 5):
    """
    Perform a hybrid search (combines vector and keyword search)
    
    Args:
        query (str): The search query
        top_k (int): Number of results to return
    """
    print(f"\n🔍 Performing Hybrid Search for: '{query}'")
    print("   Method: Combines vector similarity + BM25 keyword search")
    
    # Generate query embedding
    query_vector = get_embedding(query)
    
    # Create vector query
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=50,  # Higher K for hybrid
        fields="vector"
    )
    
    # Perform hybrid search
    results = search_client.search(
        search_text=query,  # Include text search
        vector_queries=[vector_query],
        select=["chunk_id", "chunk", "title", "parent_id"],
        top=top_k
    )
    
    print_search_results(results, "Hybrid Search")

print("✅ Hybrid search function defined!")

✅ Hybrid search function defined!


### Try Hybrid Search

Run a hybrid search with the same query to compare results:

In [9]:
# Example hybrid search
hybrid_search("What are the eligibility requirements for student loans?", top_k=3)


🔍 Performing Hybrid Search for: 'What are the eligibility requirements for student loans?'
   Method: Combines vector similarity + BM25 keyword search

📊 Hybrid Search Results

🔍 Result #1 (Score: 0.0333)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: How to qualify for private student loans 

Private student loans are primarily offered by banks, credit unions, and online lenders. They 
usually require a credit check to evaluate your creditworthiness. Most private lenders will 
let you apply with a cosigner if you don't meet their specific criter...
----------------------------------------------------------------------------------------------------

🔍 Result #2 (Score: 0.0323)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: loans are another option. Both have similar 
eligibility requirements, but they also have plenty of important differences you should 
know. 

These are the eligibility requirement for federal and private student loans: 

 Federal student loans Private student loans 

## Step 7: Hybrid Search with Semantic Reranking

**Hybrid search enhanced with AI-powered semantic reranking** for highest quality results.

### How it Works
1. First performs hybrid search (vector + keyword)
2. Then applies L2 semantic reranking using Microsoft Bing models
3. Extracts captions highlighting the most relevant passages
4. Can also provide extractive answers to questions

### Best For
- Question-answering scenarios
- Highest precision requirements
- When you need AI-generated captions and answers

In [10]:
def hybrid_search_with_semantic_reranking(query: str, top_k: int = 5):
    """
    Perform a hybrid search with semantic reranking
    
    Args:
        query (str): The search query
        top_k (int): Number of results to return
    """
    print(f"\n🔍 Performing Hybrid Search with Semantic Reranking for: '{query}'")
    print("   Method: Hybrid search + L2 semantic reranking using Microsoft Bing models")
    
    # Generate query embedding
    query_vector = get_embedding(query)
    
    # Create vector query
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=50,
        fields="vector"
    )
    
    # Perform hybrid search with semantic reranking
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["chunk_id", "chunk", "title", "parent_id"],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="semantic-config",
        query_caption=QueryCaptionType.EXTRACTIVE,
        query_answer=QueryAnswerType.EXTRACTIVE,
        top=top_k
    )
    
    print_search_results(results, "Hybrid Search with Semantic Reranking")

print("✅ Semantic reranking search function defined!")

✅ Semantic reranking search function defined!


### Try Hybrid Search with Semantic Reranking

Run semantic reranking search with the same query to see the enhanced results:

In [11]:
# Example hybrid search with semantic reranking
hybrid_search_with_semantic_reranking("What are the eligibility requirements for student loans?", top_k=3)


🔍 Performing Hybrid Search with Semantic Reranking for: 'What are the eligibility requirements for student loans?'
   Method: Hybrid search + L2 semantic reranking using Microsoft Bing models

📊 Hybrid Search with Semantic Reranking Results

🔍 Result #1 (Score: 0.0323)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: loans are another option. Both have similar 
eligibility requirements, but they also have plenty of important differences you should 
know. 

These are the eligibility requirement for federal and private student loans: 

 Federal student loans Private student loans 

Lender U.S. Department of Educat...
   💡 Caption: ...quirement for federal and private student loans:    Federal student loans Private student loans   Lender U.S. Department of Education Banks, credit unions, and online lenders   Interest rates 6.53%–9.08%, depending on loan type 3.39%–17.99%; fixed or variable      Federal student loans Private student loans   Borrowing limits  Up to $57,500 for.
   🎯 Reranker

## Step 8: Compare All Search Types

Run all three search types with the same query to compare the results and understand the differences.

In [12]:
def run_all_searches(query: str, top_k: int = 3):
    """
    Run all three types of searches for comparison
    
    Args:
        query (str): The search query
        top_k (int): Number of results to return
    """
    print("=" * 100)
    print("🚀 Azure AI Search - Query Comparison")
    print("=" * 100)
    print(f"Query: '{query}'")
    print(f"Top K Results: {top_k}")
    print(f"Index: {INDEX_NAME}")
    
    # 1. Vector Similarity Search
    vector_similarity_search(query, top_k)
    
    # 2. Hybrid Search
    hybrid_search(query, top_k)
    
    # 3. Hybrid Search with Semantic Reranking
    hybrid_search_with_semantic_reranking(query, top_k)
    
    print("\n" + "=" * 100)
    print("✅ All searches completed!")
    print("=" * 100)

print("✅ Comparison function defined!")

✅ Comparison function defined!


### Run Comparison

Execute all search types with a single query:

In [13]:
# Run comparison with a sample query
run_all_searches("How do I apply for a student loan?", top_k=3)

🚀 Azure AI Search - Query Comparison
Query: 'How do I apply for a student loan?'
Top K Results: 3
Index: student-loan-guide-index

🔍 Performing Vector Similarity Search for: 'How do I apply for a student loan?'
   Method: Pure semantic vector search using embeddings

📊 Vector Similarity Search Results

🔍 Result #1 (Score: 0.8947)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: inquiry usually 
only happens when you submit a full loan application. 

FAQ 

How do I apply for student loans step-by-step? 

Close 

To apply for student loans, start by completing the FAFSA to access federal aid. Review 
your financial aid letter from your school, and then accept any grants or f...
----------------------------------------------------------------------------------------------------

🔍 Result #2 (Score: 0.8861)
   📄 Title: StudentLoanGuide.pdf
   📝 Content: can help you get a student loan at a 
better rate than you can on your own. 

When do you need to apply for a federal student loan? 

Close 

## Step 9: Try Your Own Queries

Use the cells below to test different queries against the search index.

In [ ]:
# ============================================
# 🔧 CUSTOMIZE YOUR SEARCH HERE
# ============================================

# Enter your custom query
my_query = "What are the repayment options available?"

# Number of results to return
num_results = 3

# Choose search type: "vector", "hybrid", "semantic", or "all"
search_type = "all"

# ============================================

if search_type == "vector":
    vector_similarity_search(my_query, num_results)
elif search_type == "hybrid":
    hybrid_search(my_query, num_results)
elif search_type == "semantic":
    hybrid_search_with_semantic_reranking(my_query, num_results)
elif search_type == "all":
    run_all_searches(my_query, num_results)
else:
    print("❌ Invalid search type. Choose 'vector', 'hybrid', 'semantic', or 'all'")

## Example Queries to Try

Here are some example queries you can test:

1. "What are the eligibility requirements for student loans?"
2. "How do I apply for a student loan?"
3. "What are the repayment options available?"
4. "Can I defer my student loan payments?"
5. "What is the interest rate on student loans?"

In [ ]:
# Run multiple example queries
example_queries = [
    "Can I defer my student loan payments?",
    "What is the interest rate on student loans?"
]

for query in example_queries:
    print("\n" + "#" * 100)
    print(f"# QUERY: {query}")
    print("#" * 100)
    hybrid_search_with_semantic_reranking(query, top_k=2)

## Summary

### Search Type Comparison

| Feature | Vector Search | Hybrid Search | Hybrid + Semantic |
|---------|--------------|---------------|-------------------|
| Keyword Matching | ❌ | ✅ | ✅ |
| Semantic Understanding | ✅ | ✅ | ✅ |
| AI Reranking | ❌ | ❌ | ✅ |
| Captions | ❌ | ❌ | ✅ |
| Best For | Conceptual queries | General purpose | Q&A, High precision |

### When to Use Each

- **Vector Search**: When looking for conceptually similar content, even if exact keywords don't match
- **Hybrid Search**: Default choice for most scenarios, balances keywords and meaning
- **Semantic Reranking**: When you need the highest quality results with AI-generated insights